Here,i used fixed-size chunking for the pdf

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
PDF_PATH = "Statistics_syllabus.pdf"  
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n--- First page preview (first 500 chars) ---")
print(pages[2].page_content[:500])

C:\Users\Admin\AppData\Local\Temp\ipykernel_22980\2832660816.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
E:\Gen_AI\GEN_AI_Project\RAG_Mini_Project\Mini_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 17 pages from the PDF.

--- First page preview (first 500 chars) ---
S. Y. B.Sc. (Statistics) 
 
MES ABASAHEB GARWARE COLLEGE, PUNE 4 (AUTONOMOUS)                                                               3 
 
S.Y.B.Sc Syllabus in Statistics  
SEMESTER – III 
Paper Title: Probability Distributions – I   
 
 
Course outcomes: 
At the end of this course, students are able to… 
1)  Understand about continuous univariate and bivariate random variables, their expectation, 
variance, higher order moments and their properties.   
2)  Get the knowledge of different s


In [3]:
for i, page in enumerate(pages):
    print(f"\n========== PAGE {i+1} ==========")
    print(page.page_content)


========== PAGE 1 ==========
Maharashtra Education Society’s 
ABASAHEB GARWARE COLLEGE (AUTONOMOUS) 
KARVE ROAD, PUNE 411004 
(Affiliated to Savitribai Phule Pune University) 
 
 
Three year B. Sc. Degree Program in Statistics  
(Faculty of Science and Technology) 
 
 
Syllabus under Autonomy  
S. Y. B. Sc. (Statistics) 
 
Choice Based Credit System (C. B. C. S.) Syllabus 
To be implemented from Academic Year 2023-2024

========== PAGE 2 ==========
TITLE OF THE COURSE : S.  Y.  B.  SC. (S TATISTICS ) 
 
ELIGIBILITY:  
1.  Passed (with at least 22 credits) in F. Y. B. Sc. with Statistics as one of the subjects.  
2.  A student of the three year B. Sc. degree course wi ll not be allowed to offer Statistics 
and Statistical Techniques simultaneously in any of the three years of the course. 
 
STRUCTURE OF THE COURSE: (Each theory and practical paper has 2 credit)  
 
 
Semester  
 
Course Type  
 
Course 
Code  
 
Course Title  
 
Remark  
No. of 
Lectures  
/Practicals to be 
conducted 

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # ~150 words per chunk
    chunk_overlap=200,    # overlap keeps context at boundaries
    separators=["\n\n", "\n", ".", " "],  # tries paragraph → line → sentence → word
)

chunks = splitter.split_documents(pages)
print("Number of chunks are",len(chunks))

Number of chunks are 48


In [5]:
print(chunks[1].page_content)

TITLE OF THE COURSE : S.  Y.  B.  SC. (S TATISTICS ) 
 
ELIGIBILITY:  
1.  Passed (with at least 22 credits) in F. Y. B. Sc. with Statistics as one of the subjects.  
2.  A student of the three year B. Sc. degree course wi ll not be allowed to offer Statistics 
and Statistical Techniques simultaneously in any of the three years of the course. 
 
STRUCTURE OF THE COURSE: (Each theory and practical paper has 2 credit)  
 
 
Semester  
 
Course Type  
 
Course 
Code  
 
Course Title  
 
Remark  
No. of 
Lectures  
/Practicals to be 
conducted  
      III  
Core Course 
USST -231  Probability Distributions-I Theory  36  
USST - 232  Probability Distributions-II Theory  36  
USSTP -233  Practical Paper - I Practical  11 
Ability 
Enhancement 
course 
USLG-231  Language Theory  36  
UEVS-231  Environmental Science Theory  36  
      IV  
Core Course 
USST -241  Statistical Methods Theory  36  
USST -242  Sampling Distributions and 
Exact Tests 
 
Theory  
 
36


In [6]:
print(chunks[2].page_content)

UEVS-231  Environmental Science Theory  36  
      IV  
Core Course 
USST -241  Statistical Methods Theory  36  
USST -242  Sampling Distributions and 
Exact Tests 
 
Theory  
 
36  
USSTP -243  Practical Paper - II Practical  12 
Ability 
Enhancement 
course 
USLG-241  Language Theory  36  
UEVS-241  Environmental Science Theory  36  
GENERAL INSTRUCTIONS: Study Tour : In order to acquaint the students with applications  
of statistical methods in various fields such as in dustries, agricultural sectors, government 
institutes, etc. at least one study tour for S.Y. B .Sc. Statistics students may be arranged and study 
report should be attached in the journal.


In [7]:
#from langchain_google_genai import GoogleGenerativeAIEmbeddings or
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Embedding all chunks and storing in chroma(in memory)...")
vector_store=Chroma.from_documents(chunks,embeddings,collection_name="Statistics_Syllabus")

print("vectore_store ready.{vector_store._collection.count()}vector stored.")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1779.38it/s]


Embedding all chunks and storing in chroma(in memory)...
vectore_store ready.{vector_store._collection.count()}vector stored.


In [8]:
#Vector_Retriever

vector_retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

#test_query= test_query = "What are the distributions for which probabilities are computed using R software in Practical Paper-I?"
#tets_query="Finding summary statistics using summary and fivenum functions"

test_query = "What is the Title of experiment 9 in Practical Paper-I ?"

results = vector_retriever.invoke(test_query)

for i, doc in enumerate(results):
    print(f"\n--- Retrieved Chunk {i+1} ---")
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content)


--- Retrieved Chunk 1 ---
Page: 16
B).  Motivation for selecting the topic, abstract of the project, key-words of the project. 
C).  Text of the project : Broadly this should cover description of the sele cted problem using 
terminology in the field of application, conversion  of the problem in statistical language, 
description of collected data, small illustrative d ata set, methodology for the analysis, 
interpretation of the results, validation of the re sults, conclusions in statistical as well as 
user’s language, limitation of proposed solutions, directions for future work, references 
used, etc. 
 
 
----------xxxxx-------xxxxx-------

--- Retrieved Chunk 2 ---
Page: 1
UEVS-231  Environmental Science Theory  36  
      IV  
Core Course 
USST -241  Statistical Methods Theory  36  
USST -242  Sampling Distributions and 
Exact Tests 
 
Theory  
 
36  
USSTP -243  Practical Paper - II Practical  12 
Ability 
Enhancement 
course 
USLG-241  Language Theory  36  
UEVS-241  Environmen

In [10]:
vector_retriever = vector_store.as_retriever(
    search_kwargs={"k": 3})
test_query = "What are the distributions for which probabilities are computed using R software in Practical Paper-I?"
results = vector_retriever.invoke(test_query)

for i, doc in enumerate(results):
    print(f"\n--- Retrieved Chunk {i+1} ---")
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content)


--- Retrieved Chunk 1 ---
Page: 9
S. Y. B.Sc. (Statistics) 
 
MES ABASAHEB GARWARE COLLEGE, PUNE 4 (AUTONOMOUS)                                                               10  
 
 Paper Title - Practical Paper - I 
 
Course Outcomes: 
At the end of this course, students are able to… 
1)  Learn to fit a suitable discrete and continuous probability distributions to the data.  
2)  Identify the suitable probability model for the population.  
3)  Generate random samples from the continuous probability distributions.  
4)  Learn the basic commands of R-software. Carry out data visualization, summary statistics, 
computation of probabilities for discrete distributions using R software.  
 
Expt. No. Title of experiment 
1 Fitting of negative binomial distribution and compu tation of expected frequencies. Plot 
of expected frequencies vs. observed frequencies. 
2 Fitting of normal distribution and computation of expected frequencies. Plot of expected 
frequencies vs. observed frequencies.

In [10]:
#to see how experiment 9 was actually chunked 
print("Total fixed chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    if "Finding summary statistics" in chunk.page_content:
        print("\nFound Experiment 9!")
        print("Chunk index:", i)
        print("Page:", chunk.metadata.get("page"))
        print("\nChunk text:")
        print(chunk.page_content)
        

print("\nVectors in Chroma:", vector_store._collection.count())

Total fixed chunks: 48

Found Experiment 9!
Chunk index: 26
Page: 9

Chunk text:
8 
Diagrammatic (pie chart, bar diagram- simple, sub-d ivided, multiple) and graphical 
(histogram, stem and leaf plot, rod or spike plot, ogive curves, scatter plot) representation 
of data using R software. 
9 
Finding summary statistics using summary( ) and fivenum( ) functions.  
Calculate arithmetic mean (A.M.), geometric mean (G .M.), harmonic mean (H.M.), 
median, mode, quantiles, range, quartile deviation (Q.D.), variance, 
coefficient of variation (C.V.) (ungrouped data) using R software. 
10 Computation of probabilities for binomial, hypergeometric, Poisson, geometric, negative 
binomial and multinomial distributions using R software. 
11 Plot of p.m.f./p.d.f. and c.d.f. curve of standard discrete and continuous probability 
distributions using R software. 
 
Total no. of experiments: 11 
Each practical is of duration:  4 hours 20 minutes   
Semester – III                                       Pa

In [11]:
# ================================================================
#  EVALUATION QUESTIONS
#    SAME QUESTIONS USED FOR STRUCTURE-AWARE EVALUATION
# ================================================================

EVAL_QUERIES = [

    {
        "query": "What courses are included in Semester III?",
        "expected_page": 2
    },

    {
        "query": "What is Unit 5 of Probability Distributions-I?",
        "expected_page": 5
    },

    {
        "query": "What are the reference books for Probability Distributions-I?",
        "expected_page": 6
    },

    {
        "query": "What is Unit 3 of Probability Distributions-II?",
        "expected_page": 8
    },

    {
        "query": "What is the title of Experiment 9 in Practical Paper-I?",
        "expected_page": 10
    },

    {
        "query": "What is Unit 3 of Statistical Methods?",
        "expected_page": 12
    },

    {
        "query": "What is Unit 1 of Sampling Distributions and Exact Tests?",
        "expected_page": 14
    },

    {
        "query": "What is Experiment 9 in Practical Paper-II?",
        "expected_page": 16
    },

    {
        "query": "What are the project requirements in the syllabus?",
        "expected_page": 17
    },

    {
        "query": "What is the title of the B.Sc. Statistics program?",
        "expected_page": 1
    }
]


# ================================================================
# FUNCTION TO GET 1-BASED PDF PAGE NUMBER
# ================================================================
#
# PyPDFLoader normally stores:
#
# PDF page 1 -> metadata page 0
# PDF page 2 -> metadata page 1
# PDF page 3 -> metadata page 2
#
# Therefore:
#
# PDF page = metadata page + 1
# ================================================================

def get_fixed_page_number(doc):

    page = doc.metadata.get("page")

    if page is None:
        return None

    return int(page) + 1


# ================================================================
# CHECK PAGE METADATA
# ================================================================

print("\n" + "=" * 80)
print("FIXED-SIZE PAGE METADATA CHECK")
print("=" * 80)

for i, doc in enumerate(chunks[:10], start=1):

    print(f"\nChunk {i}")

    print("0-based metadata page:",
          doc.metadata.get("page"))

    print("1-based PDF page:",
          get_fixed_page_number(doc))

    print("Metadata:",
          doc.metadata)


# ================================================================
#  CREATE VECTOR STORE FOR FIXED-SIZE CHUNKS
# ================================================================
#
# We create a separate collection so that it does not accidentally
# mix with your structure-aware Chroma collection.
# ================================================================

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


fixed_vector_store = Chroma.from_documents(
    chunks,
    embeddings,
    collection_name="Statistics_Syllabus_Fixed_1000_200_Eval"
)


# ================================================================
#  CREATE FIXED-SIZE RETRIEVER
# ================================================================

fixed_retriever = fixed_vector_store.as_retriever(
    search_kwargs={"k": 5}
)


# ================================================================
#  EVALUATION FUNCTION
# ================================================================

def evaluate_fixed_retriever(
    retriever,
    eval_queries,
    k=5
):

    results = []

    for item in eval_queries:

        query = item["query"]

        expected_page = item["expected_page"]

        # Retrieve documents
        retrieved_docs = retriever.invoke(query)

        # Keep only top-k
        retrieved_docs = retrieved_docs[:k]

        # Convert retrieved pages to 1-based PDF pages
        retrieved_pages = [
            get_fixed_page_number(doc)
            for doc in retrieved_docs
        ]

        # Check whether expected page appears
        hit = expected_page in retrieved_pages

        # Find rank of expected page
        if hit:

            rank = retrieved_pages.index(expected_page) + 1

        else:

            rank = None

        results.append({

            "query": query,

            "expected_page": expected_page,

            "retrieved_pages": retrieved_pages,

            "hit": hit,

            "rank": rank,

            "retrieved_docs": retrieved_docs
        })

    return results


# ================================================================
# CALCULATE HIT RATE@5 AND MRR
# ================================================================

def compute_fixed_metrics(
    results,
    k=5
):

    n = len(results)

    if n == 0:

        print("No queries evaluated.")

        return

    # ------------------------------------------------------------
    # Hit Rate
    # ------------------------------------------------------------

    hits = sum(
        1
        for r in results
        if r["hit"]
    )

    hit_rate = hits / n


    # ------------------------------------------------------------
    # Reciprocal Ranks
    # ------------------------------------------------------------

    reciprocal_ranks = [

        1 / r["rank"]
        if r["hit"]
        else 0

        for r in results
    ]


    # ------------------------------------------------------------
    # MRR
    # ------------------------------------------------------------

    mrr = sum(reciprocal_ranks) / n


    # ============================================================
    # PRINT OVERALL RESULTS
    # ============================================================

    print("\n" + "=" * 80)
    print("FIXED-SIZE PAGE-LEVEL RETRIEVAL EVALUATION")
    print("=" * 80)

    print("\nNumber of queries :", n)

    print("Successful hits   :", hits)

    print(
        f"Hit Rate@{k}      : "
        f"{hit_rate:.2%}"
    )

    print(
        f"MRR               : "
        f"{mrr:.3f}"
    )


    # ============================================================
    # PRINT INDIVIDUAL RESULTS
    # ============================================================

    print("\n" + "=" * 80)
    print("INDIVIDUAL QUERY RESULTS")
    print("=" * 80)


    for r in results:

        if r["hit"]:

            status = (
                f"HIT @ rank {r['rank']}"
            )

        else:

            status = "MISS"


        print(f"\n[{status}]")

        print("Query:")

        print(r["query"])

        print(
            "Expected page:",
            r["expected_page"]
        )

        print(
            "Retrieved pages:",
            r["retrieved_pages"]
        )


# ================================================================
# RUN EVALUATION
# ================================================================

print("\n" + "=" * 80)
print("RUNNING FIXED-SIZE RETRIEVAL EVALUATION")
print("=" * 80)

results_fixed = evaluate_fixed_retriever(
    fixed_retriever,
    EVAL_QUERIES,
    k=5
)


compute_fixed_metrics(
    results_fixed,
    k=5
)


FIXED-SIZE PAGE METADATA CHECK

Chunk 1
0-based metadata page: 0
1-based PDF page: 1
Metadata: {'producer': 'Nitro Pro 12 (12.8.0.449)', 'creator': 'Nitro Pro 12 (12.8.0.449)', 'creationdate': '2023-07-09T23:33:43+05:30', 'moddate': '2023-07-09T23:33:44+05:30', 'title': 'Microsoft Word - SYBSc_Autonomous_Syllabus', 'author': 'apoorva.pali18@gmail.com', 'source': 'Statistics_syllabus.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1'}

Chunk 2
0-based metadata page: 1
1-based PDF page: 2
Metadata: {'producer': 'Nitro Pro 12 (12.8.0.449)', 'creator': 'Nitro Pro 12 (12.8.0.449)', 'creationdate': '2023-07-09T23:33:43+05:30', 'moddate': '2023-07-09T23:33:44+05:30', 'title': 'Microsoft Word - SYBSc_Autonomous_Syllabus', 'author': 'apoorva.pali18@gmail.com', 'source': 'Statistics_syllabus.pdf', 'total_pages': 17, 'page': 1, 'page_label': '2'}

Chunk 3
0-based metadata page: 1
1-based PDF page: 2
Metadata: {'producer': 'Nitro Pro 12 (12.8.0.449)', 'creator': 'Nitro Pro 12 (12.8.0.449)', 'c

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 697.44it/s]



RUNNING FIXED-SIZE RETRIEVAL EVALUATION

FIXED-SIZE PAGE-LEVEL RETRIEVAL EVALUATION

Number of queries : 10
Successful hits   : 7
Hit Rate@5      : 70.00%
MRR               : 0.508

INDIVIDUAL QUERY RESULTS

[HIT @ rank 1]
Query:
What courses are included in Semester III?
Expected page: 2
Retrieved pages: [2, 2, 1, 16, 17]

[HIT @ rank 1]
Query:
What is Unit 5 of Probability Distributions-I?
Expected page: 5
Retrieved pages: [5, 7, 9, 7, 3]

[HIT @ rank 3]
Query:
What are the reference books for Probability Distributions-I?
Expected page: 6
Retrieved pages: [9, 7, 6, 10, 2]

[MISS]
Query:
What is Unit 3 of Probability Distributions-II?
Expected page: 8
Retrieved pages: [14, 5, 14, 7, 4]

[MISS]
Query:
What is the title of Experiment 9 in Practical Paper-I?
Expected page: 10
Retrieved pages: [17, 2, 14, 6, 17]

[MISS]
Query:
What is Unit 3 of Statistical Methods?
Expected page: 12
Retrieved pages: [2, 11, 14, 11, 14]

[HIT @ rank 1]
Query:
What is Unit 1 of Sampling Distributions and E